In [6]:
!pip install sympy --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 32.6 MB/s eta 0:00:0031m36.7 MB/s eta 0:00:01
  Attempting uninstall: sympy
    Found existing installation: sympy 1.13.1
    Uninstalling sympy-1.13.1:
      Successfully uninstalled sympy-1.13.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0 requires sympy==1.13.1; python_version >= "3.9", but you have sympy 1.14.0 which is incompatible.


In [1]:
# -*- coding: utf-8 -*-
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: light
#       format_version: '1.5'
#       jupytext_version: 1.14.5 # Or your actual jupytext version
#   kernelspec:
#     display_name: Python 3 (ipykernel)
#     language: python
#     name: python3
# ---

# # Unified BERT-CRF Training for Cybersecurity NER (with Coverage Metrics)

# This notebook trains a single Named Entity Recognition (NER) model on a combined dataset containing data from multiple sources. It addresses potential issues with non-unique Sentence IDs across sources by creating a globally unique ID before processing. After training, it evaluates the model's performance on the overall test set and also on subsets corresponding to each original source, **including sentence-level coverage metrics**.
#
# **Key Changes from Original:**
# *   **Coverage Metrics:** Modifies the evaluation process to calculate and report sentence-level coverage metrics:
#     *   Average number of entities predicted per sentence.
#     *   Average number of entity types predicted per sentence.
#     *   Sentence-level entity recall (average proportion of true entities found per sentence).
#     *   Sentence-level entity type recall (average proportion of true entity types found per sentence).
# *   Explicitly uses the provided Hugging Face token.
# *   Includes fix for non-unique `Sentence_ID`s by creating a globally unique ID.
# *   Includes fix for `gather` index tensor type.
# *   Includes fix for loading checkpoints with `weights_only=False`.
# *   **Note:** Deduplication logic has been removed; this version operates on the original data potentially containing duplicate sentence content.

# ## 1. Setup and Prerequisites

# **Environment:**
# *   Ensure you have the necessary libraries installed: `torch`, `pandas`, `numpy`, `scikit-learn`, `transformers`, `tqdm`, `huggingface_hub`.
# *   Select a kernel/environment with these libraries.
# *   GPU access is recommended.

# ## 2. Imports

# +
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from transformers import BertModel, AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.utils import data
from sklearn.model_selection import train_test_split
import time
import argparse # Keep for Config structure, but won't parse CLI args
import json
from tqdm.notebook import tqdm # Use notebook-friendly tqdm
import copy
import gc # Garbage collector
import traceback # For detailed error printing
from collections import defaultdict # For coverage metrics

# Define the sources explicitly for evaluation within the test set
SOURCES_TO_EVALUATE = ['APTNER', 'CyNER', 'Attacker', 'DNRTI']
# -

# ## 3. Configuration

# **Modify these parameters as needed.**

# +
# --- Core Settings ---
BASE_OUTPUT_DIR = '/home/yasir.ech-chammakhy/lustre/cyber_cc-lcbfvhtc9qm/users/yasir.ech-chammakhy/BEACON/outputs_beacon_coverage' # Changed dir name
DATASET_PATH = '/home/yasir.ech-chammakhy/lustre/cyber_cc-lcbfvhtc9qm/users/yasir.ech-chammakhy/BEACON/dataset/beacon_stix_v1.csv' # Path to your combined dataset
SEED = 42
# --- Hugging Face Token ---
HF_TOKEN = "YOUR_HF_TOKEN_HERE" # Replace with your actual token if needed


# --- Models to Train ---
MODELS_TO_TRAIN = [
    "bert-base-cased",
    "roberta-base",
    "ehsanaghaei/SecureBERT",
    "markusbayer/CySecBERT",
    "s2w-ai/DarkBERT"
]

# --- Data Splitting ---
# These fractions apply to the original data (potentially with duplicate content)
TEST_SIZE = 0.15
VAL_SIZE = 0.15

# --- Model & Tokenizer ---
MAX_SEQ_LENGTH = 256

# --- Training Hyperparameters ---
EPOCHS = 50
BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 5e-5
LR_CRF_FC = 8e-5
WEIGHT_DECAY_FINETUNE = 1e-5
WEIGHT_DECAY_CRF_FC = 5e-6
WARMUP_PROPORTION = 0.1
MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 5

# --- Environment ---
CHECKPOINT_FREQ = 10
NUM_WORKERS = 4

# --- Derived/Fixed ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

# Set seed for reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
# -

# ## 4. Configuration Class

# +
class Config:
    def __init__(self, args_dict):
        # Model parameters
        self.model_type = args_dict['model_type']
        self.max_seq_length = args_dict['max_seq_length']
        self.batch_size = args_dict['batch_size']
        self.gradient_accumulation_steps = args_dict['gradient_accumulation_steps']
        self.total_train_epochs = args_dict['epochs']
        self.output_dir = args_dict['output_dir']
        self.learning_rate = args_dict['learning_rate']
        self.lr_crf_fc = args_dict['lr_crf_fc']
        self.weight_decay_crf_fc = args_dict['weight_decay_crf_fc']
        self.weight_decay_finetune = args_dict['weight_decay_finetune']
        self.warmup_proportion = args_dict['warmup_proportion']
        self.early_stopping_patience = args_dict['early_stopping_patience']
        self.checkpoint_freq = args_dict['checkpoint_freq']
        self.device = args_dict['device']
        self.seed = args_dict['seed']
        self.max_grad_norm = args_dict['max_grad_norm']
        self.test_size = args_dict['test_size']
        self.val_size = args_dict['val_size']
        self.num_workers = args_dict['num_workers']
        self.dataset_path = args_dict['dataset_path']
        self.hf_token = args_dict['hf_token']

        os.makedirs(self.output_dir, exist_ok=True)
# -

# ## 5. Data Processing Classes & Functions (Original + Coverage Support)

# +
class InputExample:
    """A single training/test example for token classification."""
    def __init__(self, guid, words, labels, source):
        self.guid = guid # Unique identifier for the example instance
        self.words = words
        self.labels = labels
        self.source = source # Store the source dataset name

class InputFeatures:
    """A single set of features of data."""
    def __init__(self, guid, input_ids, input_mask, segment_ids, predict_mask, label_ids):
        self.guid = guid # Carry GUID into features
        self.input_ids = input_ids
        self.input_mask = input_mask
        self.segment_ids = segment_ids
        self.predict_mask = predict_mask
        self.label_ids = label_ids

# ---------------------------------------------------------
# prepare_unified_dataset function (Original Version - No Deduplication)
# ---------------------------------------------------------
def prepare_unified_dataset(df):
    """
    Processes the entire DataFrame to create InputExamples for all sentences,
    storing the source for each example and creating a globally unique ID.
    """
    start_time = time.time()
    print("--- Starting Data Preparation (No Deduplication) ---")

    # --- 1. Create globally unique sentence ID ---
    print("Creating globally unique sentence IDs (Source_SentenceID)...")
    df['Sentence_ID'] = df['Sentence_ID'].fillna('unknown_id').astype(str)
    df['Source'] = df['Source'].fillna('unknown_source').astype(str)
    df['Word'] = df['Word'].fillna('').astype(str)
    df['STIX_Tag'] = df['STIX_Tag'].fillna('O').astype(str)
    df['Unique_Sentence_ID'] = df['Source'] + '_' + df['Sentence_ID']
    print("Unique instance IDs created.")

    # --- 2. Group by Unique_Sentence_ID and create InputExamples ---
    grouped = df.groupby('Unique_Sentence_ID')
    initial_sentence_count = len(grouped)
    print(f"Found {initial_sentence_count} initial sentence instances based on Unique_Sentence_ID.")
    print("Creating InputExamples...")

    examples = []
    # Use notebook tqdm
    for unique_sentence_id, group in tqdm(grouped, desc="Preparing Examples"):
        if group.empty:
            print(f"Warning: Unique_Sentence_ID {unique_sentence_id} resulted in an empty group. Skipping.")
            continue

        words = group['Word'].tolist()
        labels = group['STIX_Tag'].tolist()
        source = group['Source'].iloc[0]

        # Handle potential empty sentences (e.g., if only '.' was present and filtered)
        if not words:
            # print(f"Warning: Skipping empty sentence for GUID {unique_sentence_id}")
            continue

        guid = unique_sentence_id # Use the unique ID as the GUID
        examples.append(InputExample(guid=guid, words=words, labels=labels, source=source))


    end_time = time.time()
    print(f"Created {len(examples)} InputExamples.")
    print(f"Data preparation took {(end_time - start_time):.2f} seconds.")
    print("--- Data Preparation Finished ---")
    return examples
# ---------------------------------------------------------
# End of prepare_unified_dataset
# ---------------------------------------------------------


def example2feature(example, tokenizer, label_map, max_seq_length):
    """Converts a single `InputExample` into an `InputFeatures`."""
    add_label = 'X'
    tokens = []
    label_ids = []
    predict_mask = []

    tokens.append(tokenizer.cls_token)
    label_ids.append(label_map.get('[CLS]', label_map['O']))
    predict_mask.append(0)

    for i, word in enumerate(example.words):
        word_tokenized = tokenizer.tokenize(str(word))
        if not word_tokenized: word_tokenized = [tokenizer.unk_token]

        tokens.extend(word_tokenized)
        label = example.labels[i] if i < len(example.labels) else 'O'
        for j, sub_token in enumerate(word_tokenized):
            if j == 0:
                label_ids.append(label_map.get(label, label_map['O']))
                predict_mask.append(1)
            else:
                label_ids.append(label_map.get(add_label, label_map['O']))
                predict_mask.append(0)

    special_tokens_count = tokenizer.num_special_tokens_to_add()
    if len(tokens) > max_seq_length - special_tokens_count:
        tokens = tokens[:(max_seq_length - special_tokens_count)]
        label_ids = label_ids[:(max_seq_length - special_tokens_count)]
        predict_mask = predict_mask[:(max_seq_length - special_tokens_count)]

    tokens.append(tokenizer.sep_token)
    label_ids.append(label_map.get('[SEP]', label_map['O']))
    predict_mask.append(0)

    input_ids = tokenizer.convert_tokens_to_ids(tokens)
    input_mask = [1] * len(input_ids)
    segment_ids = [0] * len(input_ids)

    if not (len(input_ids) == len(input_mask) == len(segment_ids) == len(label_ids) == len(predict_mask)):
        print(f"Length mismatch ERROR for GUID {example.guid}!")
        return None

    # Store GUID in the feature itself
    return InputFeatures(
        guid=example.guid,
        input_ids=input_ids,
        input_mask=input_mask,
        segment_ids=segment_ids,
        predict_mask=predict_mask,
        label_ids=label_ids
    )

class NerDataset(data.Dataset):
    """Dataset wrapping features and providing GUIDs."""
    def __init__(self, examples, tokenizer, label_map, max_seq_length):
        self.tokenizer = tokenizer
        self.label_map = label_map
        self.max_seq_length = max_seq_length
        # Store examples temporarily to create features
        self.features = self._create_features(examples)
        # Create a mapping from feature index to GUID for easy lookup during eval
        self.idx2guid = {i: feat.guid for i, feat in enumerate(self.features)}
        # Can now optionally delete examples if memory is tight
        # del examples
        # gc.collect()

    def _create_features(self, examples): # Accept examples as argument
        features = []
        print(f"Converting {len(examples)} examples to features...")
        # Use notebook tqdm
        for example in tqdm(examples, desc="Creating Features"):
            try:
                feat = example2feature(example, self.tokenizer, self.label_map, self.max_seq_length)
                if feat is not None:
                    features.append(feat)
                else:
                    print(f"Warning: Failed to convert example GUID {example.guid} to features. Skipping.")
            except Exception as e:
                print(f"Error converting example GUID {example.guid} to features: {e}. Skipping.")
                traceback.print_exc()
        print(f"Successfully created {len(features)} features.")
        return features

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        feat = self.features[idx]
        # Return elements including the index to map back to GUID later
        return (
            feat.input_ids,
            feat.input_mask,
            feat.segment_ids,
            feat.predict_mask,
            feat.label_ids,
            idx # Return index
        )

    @staticmethod
    def pad(batch):
        """Pads sequences within a batch to the maximum sequence length."""
        seqlen_list = [len(sample[0]) for sample in batch]
        maxlen = max(seqlen_list) if seqlen_list else 0
        pad_label_id = 0

        # Pad data features
        f_data = lambda x, seqlen, pad_value: [sample[x] + [pad_value] * (seqlen - len(sample[x])) for sample in batch]
        input_ids_list = torch.LongTensor(f_data(0, maxlen, 0))
        input_mask_list = torch.LongTensor(f_data(1, maxlen, 0))
        segment_ids_list = torch.LongTensor(f_data(2, maxlen, 0))
        predict_mask_int = f_data(3, maxlen, 0)
        predict_mask_list = torch.BoolTensor(predict_mask_int)
        label_ids_list = torch.LongTensor(f_data(4, maxlen, pad_label_id))

        # Collect indices (don't pad these)
        indices = [sample[5] for sample in batch]
        indices_tensor = torch.LongTensor(indices)

        return input_ids_list, input_mask_list, segment_ids_list, predict_mask_list, label_ids_list, indices_tensor


# --- BIO/STIX Tag Decoding Helper ---
def decode_tags_to_entities(tags):
    """
    Decodes a list of BIO-prefixed tags (like 'B-Malware', 'I-Malware', 'O')
    or non-prefixed tags (treated as B) into a list of entity spans.
    Ignores 'X', '[CLS]', '[SEP]' tags.

    Args:
        tags (list): A list of tag strings.

    Returns:
        list: A list of tuples, where each tuple represents an entity:
              (entity_type, start_index, end_index).
              Indices are inclusive word indices in the original sentence.
    """
    entities = []
    current_entity = None
    ignore_tags_for_entity = {'O', 'X', '[CLS]', '[SEP]'}

    for i, tag_str in enumerate(tags):
        tag_str = str(tag_str) # Ensure string
        bio_tag = 'O'
        entity_type = None

        if tag_str in ignore_tags_for_entity:
             bio_tag = 'O'
        elif '-' in tag_str:
            parts = tag_str.split('-', 1)
            if len(parts) == 2 and parts[0] in ['B', 'I']:
                bio_tag = parts[0]
                entity_type = parts[1]
            else: # Malformed tag like "Malware-Tool" treat as B
                bio_tag = 'B'
                entity_type = tag_str
        else: # Tag without BIO prefix like "Malware", treat as B
             bio_tag = 'B'
             entity_type = tag_str

        is_start_of_entity = bio_tag == 'B'
        is_inside_entity = bio_tag == 'I'
        is_outside_entity = bio_tag == 'O'

        if is_outside_entity:
            if current_entity: # End of current entity
                entities.append(current_entity)
            current_entity = None
        elif is_start_of_entity:
            if current_entity: # End previous entity if any
                entities.append(current_entity)
            # Start new entity
            current_entity = (entity_type, i, i)
        elif is_inside_entity:
            if current_entity and entity_type == current_entity[0]:
                # Extend current entity
                current_entity = (current_entity[0], current_entity[1], i)
            else: # I tag doesn't match or no current entity, treat as B
                if current_entity:
                    entities.append(current_entity)
                # print(f"Warning: Treating I-{entity_type} at index {i} as B-{entity_type}")
                current_entity = (entity_type, i, i)

    # Add the last entity if it exists
    if current_entity:
        entities.append(current_entity)

    return entities


# --- Model Utilities ---
def log_sum_exp_batch(log_tensor, axis=-1):
    """ Calculates log_sum_exp in a numerically stable way for a batch. """
    # Simplified check: if empty, return -inf shaped tensor
    if log_tensor.nelement() == 0:
        out_shape = list(log_tensor.shape)
        if axis is not None and abs(axis) < len(out_shape):
            del out_shape[axis]
        return torch.full(out_shape, -float('inf'), device=log_tensor.device, dtype=log_tensor.dtype)

    max_score = torch.max(log_tensor, axis, keepdim=True)[0]
    max_score_adjusted = max_score.masked_fill(torch.isneginf(max_score), 0.0) # Adjust -inf for subtraction
    sum_exp = torch.exp(log_tensor - max_score_adjusted).sum(axis, keepdim=True)
    # Handle cases where sum_exp might be zero due to underflow
    log_sum_exp_val = torch.log(sum_exp + 1e-10) + max_score_adjusted # Add small epsilon for stability
    # Re-apply -inf where max_score was -inf
    log_sum_exp_val = log_sum_exp_val.masked_fill(torch.isneginf(max_score), -float('inf'))

    return log_sum_exp_val.squeeze(axis)


# --- Evaluation Function (Modified for Coverage) ---
def evaluate(model, dataloader, epoch, dataset_name, label_map, idx2label, device):
    """
    Evaluates the model, calculating standard P/R/F1 and sentence-level coverage metrics.
    """
    model.eval()
    start = time.time()

    # Store results per sentence for coverage calculation
    sentence_results = []
    # Store valid predictions/labels for overall P/R/F1 calculation
    all_valid_preds_flat = []
    all_valid_labels_flat = []

    with torch.no_grad():
        pbar = tqdm(dataloader, desc=f"Evaluating on {dataset_name}", leave=False)
        for batch in pbar:
            # Unpack batch including the indices tensor
            input_ids, input_mask, segment_ids, predict_mask, label_ids, indices = batch
            input_ids, input_mask, segment_ids, predict_mask, label_ids = \
                input_ids.to(device), input_mask.to(device), segment_ids.to(device), \
                predict_mask.to(device), label_ids.to(device)

            if input_ids.size(0) == 0: continue

            try:
                 _, predicted_label_seq_ids = model(input_ids, segment_ids, input_mask)
            except Exception as eval_err:
                 print(f"\nError during model forward pass in evaluation: {eval_err}")
                 continue # Skip batch on error

            # Process each sentence in the batch
            for i in range(input_ids.size(0)):
                # Use input_mask to determine actual sequence length (excluding padding)
                seq_len_actual = input_mask[i].sum().item()
                if seq_len_actual == 0: continue # Skip empty sequences

                # Get predictions and labels for the actual sequence length
                preds_i_all = predicted_label_seq_ids[i][:seq_len_actual]
                labels_i_all = label_ids[i][:seq_len_actual]
                pred_mask_i_bool = predict_mask[i][:seq_len_actual] # Boolean mask

                # Select predictions and labels where predict_mask is True (first subword)
                valid_indices = pred_mask_i_bool.nonzero(as_tuple=False).squeeze(-1)

                if valid_indices.numel() > 0:
                    valid_labels_ids = labels_i_all[valid_indices]
                    valid_preds_ids = preds_i_all[valid_indices]

                    # Store for overall P/R/F1
                    all_valid_labels_flat.extend(valid_labels_ids.cpu().tolist())
                    all_valid_preds_flat.extend(valid_preds_ids.cpu().tolist())

                    # --- Sentence-level Coverage Calculation ---
                    # Convert valid IDs to tags (strings)
                    true_tags = [idx2label.get(l_id, 'O') for l_id in valid_labels_ids.cpu().tolist()]
                    pred_tags = [idx2label.get(p_id, 'O') for p_id in valid_preds_ids.cpu().tolist()]

                    # Decode tags to entity spans (type, start_idx, end_idx)
                    true_entities = decode_tags_to_entities(true_tags)
                    pred_entities = decode_tags_to_entities(pred_tags)

                    # Calculate correct entities (exact match on type and span)
                    correct_entities = set(true_entities) & set(pred_entities)

                    # Calculate entity types
                    true_entity_types = {e[0] for e in true_entities} # Set of unique types
                    pred_entity_types = {e[0] for e in pred_entities}
                    correct_entity_types = {e[0] for e in correct_entities}

                    # Get the original GUID for this sentence using the index from the batch
                    original_dataset_index = indices[i].item()
                    guid = dataloader.dataset.idx2guid.get(original_dataset_index, f"unknown_idx_{original_dataset_index}")

                    sentence_results.append({
                        'guid': guid,
                        'num_true_entities': len(true_entities),
                        'num_pred_entities': len(pred_entities),
                        'num_correct_entities': len(correct_entities),
                        'num_true_entity_types': len(true_entity_types),
                        'num_pred_entity_types': len(pred_entity_types),
                        'num_correct_entity_types': len(correct_entity_types)
                    })
                # else: sentence had no valid first-subword tokens (e.g., only CLS/SEP/PAD)


    end = time.time()
    eval_duration = (end - start) / 60.0

    # --- Aggregate Results ---
    # 1. Overall P/R/F1 (using calculate_metrics)
    overall_precision, overall_recall, overall_f1, class_metrics = 0.0, 0.0, 0.0, {}
    if all_valid_labels_flat:
         try:
             # Ensure calculate_metrics ignores 'O', 'X', '[CLS]', '[SEP]'
             overall_precision, overall_recall, overall_f1, class_metrics = calculate_metrics(
                 np.array(all_valid_labels_flat),
                 np.array(all_valid_preds_flat),
                 label_map,
                 idx2label
             )
         except Exception as metrics_err:
             print(f"\nError calculating overall metrics: {metrics_err}")
             traceback.print_exc()
    else:
        print("\nWarning: No valid tokens found for overall evaluation. Returning zero metrics.")


    # 2. Coverage Metrics
    total_sentences_evaluated = len(sentence_results)
    sum_true_entities = 0
    sum_pred_entities = 0
    sum_correct_entities = 0
    sum_true_entity_types = 0
    sum_pred_entity_types = 0
    sum_correct_entity_types = 0
    sum_sentence_entity_recall = 0.0
    sum_sentence_type_recall = 0.0
    sentences_with_true_entities = 0
    sentences_with_true_types = 0

    if total_sentences_evaluated > 0:
        for res in sentence_results:
            sum_true_entities += res['num_true_entities']
            sum_pred_entities += res['num_pred_entities']
            sum_correct_entities += res['num_correct_entities']
            sum_true_entity_types += res['num_true_entity_types']
            sum_pred_entity_types += res['num_pred_entity_types']
            sum_correct_entity_types += res['num_correct_entity_types']

            # Calculate sentence-level recall contributions (Macro-average)
            # Handle division by zero: if num_true is 0, recall is 1 if num_pred is also 0, else 0.
            if res['num_true_entities'] > 0:
                sum_sentence_entity_recall += res['num_correct_entities'] / res['num_true_entities']
                sentences_with_true_entities += 1
            elif res['num_pred_entities'] == 0: # Correctly handles 0/0 -> recall 1
                sum_sentence_entity_recall += 1.0
            # else: num_true is 0 but num_pred > 0, recall is 0 (already added 0)

            if res['num_true_entity_types'] > 0:
                sum_sentence_type_recall += res['num_correct_entity_types'] / res['num_true_entity_types']
                sentences_with_true_types += 1
            elif res['num_pred_entity_types'] == 0: # Correctly handles 0/0 -> recall 1
                sum_sentence_type_recall += 1.0
            # else: num_true_types is 0 but num_pred_types > 0, recall is 0

        # Calculate final averages
        avg_pred_entities_per_sentence = sum_pred_entities / total_sentences_evaluated
        avg_pred_types_per_sentence = sum_pred_entity_types / total_sentences_evaluated

        # Macro Average Sentence Recall: Average of each sentence's recall
        sentence_entity_recall_macro = sum_sentence_entity_recall / total_sentences_evaluated if total_sentences_evaluated > 0 else 0.0
        sentence_type_recall_macro = sum_sentence_type_recall / total_sentences_evaluated if total_sentences_evaluated > 0 else 0.0

        # Micro Average Sentence Recall: Total correct entities / Total true entities
        sentence_entity_recall_micro = sum_correct_entities / sum_true_entities if sum_true_entities > 0 else (1.0 if sum_pred_entities == 0 else 0.0)
        # Micro Average Type Recall: Total correct types / Total true types (This is less standard, sum_correct_entity_types is tricky)
        # Let's focus on entity micro recall. Total correct types isn't directly comparable across sentences.
        # Micro Type Recall approximation: Sum of correct unique types identified across all sentences / Sum of unique true types across all sentences
        # This is still tricky. Sticking to entity micro recall is safer.
        sentence_type_recall_micro = sum_correct_entity_types / sum_true_entity_types if sum_true_entity_types > 0 else (1.0 if sum_pred_entity_types == 0 else 0.0) # Approximation

    else: # No sentences evaluated
        avg_pred_entities_per_sentence = 0.0
        avg_pred_types_per_sentence = 0.0
        sentence_entity_recall_macro = 0.0
        sentence_type_recall_macro = 0.0
        sentence_entity_recall_micro = 0.0
        sentence_type_recall_micro = 0.0 # Approximation

    print(f'\n--- Evaluation Results ({dataset_name} at {epoch}) ---')
    print(f'Eval Time: {eval_duration:.3f} minutes, Sentences Evaluated: {total_sentences_evaluated}')
    print('--- Overall Performance (Token Level) ---')
    print(f'Overall Precision: {100.*overall_precision:.2f}%')
    print(f'Overall Recall: {100.*overall_recall:.2f}%')
    print(f'Overall F1-Score: {100.*overall_f1:.2f}%')
    print('--- Coverage Metrics (Sentence Level) ---')
    print(f'Avg Predicted Entities / Sentence: {avg_pred_entities_per_sentence:.3f}')
    print(f'Avg Predicted Entity Types / Sentence: {avg_pred_types_per_sentence:.3f}')
    print(f'Sentence Entity Recall (Macro Avg): {100.*sentence_entity_recall_macro:.2f}%')
    print(f'Sentence Type Recall (Macro Avg): {100.*sentence_type_recall_macro:.2f}%')
    print(f'Sentence Entity Recall (Micro Avg - Total Correct/Total True): {100.*sentence_entity_recall_micro:.2f}%')
    # print(f'Sentence Type Recall (Micro Avg - Approx): {100.*sentence_type_recall_micro:.2f}%') # Micro type recall is less standard
    print('--------------------------------------------------------------')

    # Per-class metrics (based on overall token counts)
    if idx2label and class_metrics:
        print("Per-class metrics (P, R, F1, Support - based on overall tokens):")
        sorted_class_metrics = sorted(class_metrics.items(), key=lambda item: item[1]['label'])
        for cls_id, metrics in sorted_class_metrics:
            label = metrics['label']
            p = float(metrics['precision'])
            r = float(metrics['recall'])
            f = float(metrics['f1'])
            s = int(metrics['support'])
            print(f"  {label:<20}: P={p:.4f}, R={r:.4f}, F1={f:.4f}, S={s}")
        print('--------------------------------------------------------------')

    coverage_results = {
        "avg_pred_entities_per_sentence": avg_pred_entities_per_sentence,
        "avg_pred_types_per_sentence": avg_pred_types_per_sentence,
        "sentence_entity_recall_macro": sentence_entity_recall_macro,
        "sentence_type_recall_macro": sentence_type_recall_macro,
        "sentence_entity_recall_micro": sentence_entity_recall_micro,
        # "sentence_type_recall_micro": sentence_type_recall_micro, # Optional micro type recall
        "total_sentences_evaluated": total_sentences_evaluated,
        "total_true_entities": sum_true_entities,
        "total_pred_entities": sum_pred_entities,
        "total_correct_entities": sum_correct_entities,
        "total_true_entity_types_sum": sum_true_entity_types, # Sum across sentences
        "total_pred_entity_types_sum": sum_pred_entity_types,
        "total_correct_entity_types_sum": sum_correct_entity_types
    }

    # Return standard F1, class metrics, and the new coverage dictionary
    return overall_f1, class_metrics, coverage_results


# --- calculate_metrics function (Unchanged) ---
def calculate_metrics(y_true, y_pred, label_map, idx2label=None):
    """ Calculates overall P, R, F1 and per-class metrics, ignoring specified labels. """
    if idx2label is None: idx2label = {v: k for k, v in label_map.items()}

    ignore_labels = {'O', 'X', '[CLS]', '[SEP]'}
    ignore_ids = {label_map[label] for label in ignore_labels if label in label_map}

    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()

    true_valid_mask = np.isin(y_true, list(ignore_ids), invert=True)
    pred_valid_mask = np.isin(y_pred, list(ignore_ids), invert=True)

    num_gold = np.sum(true_valid_mask)
    correct_mask = np.logical_and(y_true == y_pred, true_valid_mask)
    num_correct = np.sum(correct_mask)
    num_proposed = np.sum(pred_valid_mask)

    precision = num_correct / num_proposed if num_proposed > 0 else 0.0
    recall = num_correct / num_gold if num_gold > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    class_metrics = {}
    unique_relevant_true_ids = np.unique(y_true[true_valid_mask])
    unique_relevant_pred_ids = np.unique(y_pred[pred_valid_mask])
    all_relevant_class_ids = sorted(list(set(unique_relevant_true_ids) | set(unique_relevant_pred_ids)))
    unique_relevant_classes = [cls_id for cls_id in all_relevant_class_ids if cls_id not in ignore_ids]

    for cls_id in unique_relevant_classes:
        cls_label = idx2label.get(cls_id, f"Unknown_{cls_id}")
        tp_mask = np.logical_and(y_true == cls_id, y_pred == cls_id)
        fp_mask = np.logical_and(y_true != cls_id, y_pred == cls_id)
        fn_mask = np.logical_and(y_true == cls_id, y_pred != cls_id)

        tp = np.sum(tp_mask)
        fp = np.sum(fp_mask)
        fn = np.sum(fn_mask)

        cls_precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        cls_recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        cls_f1 = 2 * cls_precision * cls_recall / (cls_precision + cls_recall) if (cls_precision + cls_recall) > 0 else 0.0
        support = tp + fn

        class_metrics[cls_id] = {
            'label': cls_label, 'precision': cls_precision, 'recall': cls_recall,
            'f1': cls_f1, 'support': support
        }
    return precision, recall, f1, class_metrics
# -

# ## 6. Model Definition (BERT-CRF)

# +
# Model Definition remains the same as the original code
class BERT_CRF_NER(nn.Module):
    def __init__(self, model_type, start_label_id, stop_label_id, num_labels, device, hf_token=None):
        super(BERT_CRF_NER, self).__init__()
        self.num_labels = num_labels
        self.start_label_id = start_label_id
        self.stop_label_id = stop_label_id
        self.device = device
        self.model_type = model_type
        self.hf_token = hf_token

        try:
            print(f"Initializing encoder: {model_type}")
            self.encoder = AutoModel.from_pretrained(model_type, token=self.hf_token)
        except Exception as e:
             print(f"FATAL: Error loading model '{model_type}': {e}")
             if "authentication" in str(e).lower(): print("Check HF Token validity.")
             traceback.print_exc(); raise
        self.hidden_size = self.encoder.config.hidden_size
        print(f"Encoder loaded. Hidden size: {self.hidden_size}")

        self.dropout = nn.Dropout(0.2)
        self.hidden2label = nn.Linear(self.hidden_size, self.num_labels)
        self.transitions = nn.Parameter(torch.randn(self.num_labels, self.num_labels))
        self.transitions.data[start_label_id, :] = -10000.0
        self.transitions.data[:, stop_label_id] = -10000.0
        nn.init.xavier_uniform_(self.hidden2label.weight)
        nn.init.constant_(self.hidden2label.bias, 0.0)

    def _get_encoder_features(self, input_ids, segment_ids, input_mask):
        model_type_lower = self.model_type.lower()
        if 'roberta' in model_type_lower:
             encoder_output = self.encoder(input_ids=input_ids, attention_mask=input_mask)
        else:
             encoder_output = self.encoder(input_ids=input_ids, token_type_ids=segment_ids, attention_mask=input_mask)

        if hasattr(encoder_output, 'last_hidden_state'): sequence_output = encoder_output.last_hidden_state
        elif isinstance(encoder_output, (tuple, list)): sequence_output = encoder_output[0]
        else: print("Warning: Unexpected encoder output format."); sequence_output = encoder_output

        sequence_output = self.dropout(sequence_output)
        emission_scores = self.hidden2label(sequence_output)
        return emission_scores

    def _forward_alg(self, feats, mask):
        batch_size, seq_len, num_labels = feats.shape
        mask_bool = mask.bool()
        log_alpha = torch.full((batch_size, num_labels), -10000.0, device=self.device)
        log_alpha[:, self.start_label_id] = 0.0
        transitions = self.transitions.unsqueeze(0)

        for t in range(seq_len):
            mask_t = mask_bool[:, t].unsqueeze(1)
            emit_scores_t = feats[:, t]
            log_alpha_expanded = log_alpha.unsqueeze(2)
            emit_scores_t_expanded = emit_scores_t.unsqueeze(1)
            scores_t = log_alpha_expanded + transitions + emit_scores_t_expanded
            next_log_alpha = log_sum_exp_batch(scores_t, axis=1)
            log_alpha = torch.where(mask_t, next_log_alpha, log_alpha)

        log_alpha += self.transitions[:, self.stop_label_id].unsqueeze(0)
        partition = log_sum_exp_batch(log_alpha, axis=1)
        return partition

    # Fix for gather index type
    def _score_sentence(self, feats, label_ids, mask):
        batch_size, seq_len, _ = feats.shape
        mask_float = mask.float(); mask_bool = mask.bool()
        score = torch.zeros(batch_size, device=self.device)
        first_labels = label_ids[:, 0]
        score += self.transitions[self.start_label_id, first_labels]

        for t in range(seq_len):
            mask_t = mask_bool[:, t]
            current_labels = label_ids[:, t]
            emit_scores = feats[:, t].gather(1, current_labels.unsqueeze(1).long()).squeeze(1) # Cast to long
            score += emit_scores * mask_t.float()
            if t < seq_len - 1:
                next_labels = label_ids[:, t + 1]
                transition_scores = self.transitions[current_labels.long(), next_labels.long()] # Cast to long
                mask_next_t = mask_bool[:, t + 1]
                score += transition_scores * mask_next_t.float()

        seq_lengths = mask_float.sum(dim=1).long()
        last_valid_idx = (seq_lengths - 1).clamp(min=0)
        last_labels = label_ids.gather(1, last_valid_idx.unsqueeze(1).long()).squeeze(1) # Cast to long
        score += self.transitions[last_labels.long(), self.stop_label_id] # Cast to long
        return score

    def _viterbi_decode(self, feats, mask):
        batch_size, seq_len, num_labels = feats.shape
        mask_float = mask.float(); mask_bool = mask.bool()
        log_delta = torch.full((batch_size, num_labels), -10000.0, device=self.device)
        log_delta[:, self.start_label_id] = 0.0
        psi = torch.zeros((batch_size, seq_len, num_labels), dtype=torch.long, device=self.device)
        transitions_expanded = self.transitions.unsqueeze(0)

        for t in range(seq_len):
            mask_t = mask_bool[:, t].unsqueeze(1)
            if not mask_t.any(): continue
            emit_scores_t = feats[:, t]
            scores_t = log_delta.unsqueeze(2) + transitions_expanded
            max_scores_t, max_indices_t = torch.max(scores_t, dim=1)
            max_scores_t += emit_scores_t
            psi[:, t, :] = max_indices_t
            log_delta = torch.where(mask_t, max_scores_t, log_delta)

        log_delta += self.transitions[:, self.stop_label_id].unsqueeze(0)
        best_paths = torch.zeros((batch_size, seq_len), dtype=torch.long, device=self.device)
        best_scores, last_tags = torch.max(log_delta, dim=1)

        for b in range(batch_size):
            seq_len_b = int(mask_float[b].sum().item())
            if seq_len_b == 0: continue
            best_paths[b, seq_len_b - 1] = last_tags[b]
            for t in range(seq_len_b - 2, -1, -1):
                 current_tag_idx_at_t_plus_1 = best_paths[b, t + 1]
                 best_paths[b, t] = psi[b, t + 1, current_tag_idx_at_t_plus_1]

        return best_scores, best_paths

    def neg_log_likelihood(self, input_ids, segment_ids, input_mask, label_ids):
        mask = input_mask.float()
        feats = self._get_encoder_features(input_ids, segment_ids, input_mask)
        forward_score = self._forward_alg(feats, mask)
        gold_score = self._score_sentence(feats, label_ids, mask)
        loss = forward_score - gold_score

        if torch.isnan(loss).any() or torch.isinf(loss).any():
             print(f"\nWarning: NaN/Inf detected in loss. Skipping mean calculation for safety.")
             # Returning a tensor that signals an issue might be better than returning 0
             return torch.tensor(float('nan'), device=self.device) # Or raise an error

        return torch.mean(loss)

    def forward(self, input_ids, segment_ids, input_mask):
        mask = input_mask.float()
        feats = self._get_encoder_features(input_ids, segment_ids, input_mask)
        best_scores, best_paths = self._viterbi_decode(feats, mask)
        return best_scores, best_paths
# -

# ## 7. Prediction Function

# +
# Uses the same BIO decoding logic now present in evaluate()
def predict_entities(model, tokenizer, text, label_map, idx2label, max_seq_length, device):
    """ Predicts entities in a given text string using the trained model. """
    model.eval()
    model.to(device)

    if not text or not text.strip(): return []
    words = text.split() # Simple split, might need refinement
    if not words: return []

    predict_example = InputExample(guid="predict_0", words=words, labels=['O'] * len(words), source='predict')
    features = None
    try:
        features = example2feature(predict_example, tokenizer, label_map, max_seq_length)
        if features is None:
            print("Error: Failed to create features for prediction.")
            return []

        input_ids = torch.LongTensor([features.input_ids]).to(device)
        input_mask = torch.LongTensor([features.input_mask]).to(device)
        segment_ids = torch.LongTensor([features.segment_ids]).to(device)
        predict_mask_batch = torch.BoolTensor([features.predict_mask]).to(device)

    except Exception as e:
        print(f"Error creating features or tensors for prediction: {e}")
        return []

    tag_seq = None
    try:
        with torch.no_grad():
            _, tag_seq = model(input_ids, segment_ids, input_mask)
    except Exception as e:
        print(f"Error during model prediction forward pass: {e}")
        return []

    if tag_seq is None or tag_seq.nelement() == 0:
        print("Prediction failed, no tag sequence generated.")
        return []

    # --- Post-process Predictions ---
    valid_pred_tags = []
    p_mask = predict_mask_batch[0].cpu().numpy()
    t_seq = tag_seq[0].cpu().numpy()
    feat_len = len(p_mask)
    start_index = 1 # Skip [CLS]

    for i in range(start_index, feat_len):
        if i < len(t_seq) and p_mask[i]:
            tag_id = t_seq[i]
            label = idx2label.get(tag_id, 'O')
            valid_pred_tags.append(label)

    # Align predictions with original words (handle truncation)
    num_words = len(words)
    num_preds = len(valid_pred_tags)
    aligned_tags = valid_pred_tags
    if num_preds != num_words:
        min_len = min(num_words, num_preds)
        print(f"Warning: Word/prediction mismatch ({num_words} words, {num_preds} preds). Aligning to {min_len}.")
        words = words[:min_len]
        aligned_tags = valid_pred_tags[:min_len]
        if min_len == 0: return []

    # --- Convert BIO tags to entities ---
    decoded_spans = decode_tags_to_entities(aligned_tags) # Get (type, start, end) indices

    entities = []
    for entity_type, start_idx, end_idx in decoded_spans:
         entity_text = " ".join(words[start_idx : end_idx + 1])
         entities.append({
             'text': entity_text,
             'start_token': start_idx,
             'end_token': end_idx,
             'type': entity_type
         })

    return entities
# -

# ## 8. Training Function (Updated for Coverage Metrics)

# +
def train(config: Config, test_examples): # Accept test examples from the outer loop
    """ Trains the unified BERT-CRF model and evaluates it, including coverage metrics. """
    print(f"\n===== Starting Training Run for Model: {config.model_type} =====")
    # Use the specific output dir from config
    model_base_dir = os.path.join(config.output_dir, config.model_type.replace('/', '_') + "_unified_coverage")
    os.makedirs(model_base_dir, exist_ok=True)
    print(f"Output Directory: {model_base_dir}")

    # --- Load Data (Original approach - prepare ALL examples first) ---
    print(f"Loading full unified dataset from {config.dataset_path} (for train/dev split)...")
    try:
        df_full = pd.read_csv(config.dataset_path, keep_default_na=False, encoding='utf-8', on_bad_lines='warn', dtype=str)
        # Use the original prepare function (no deduplication)
        all_examples = prepare_unified_dataset(df_full)
        del df_full; gc.collect()
        if not all_examples:
            raise ValueError("prepare_unified_dataset returned no examples.")
    except FileNotFoundError:
        print(f"Error: Dataset file not found at {config.dataset_path}"); return False, None
    except Exception as e:
        print(f"Error loading or preparing full dataset: {e}"); traceback.print_exc(); return False, None

    # --- Create Label Map (from ALL potentially duplicated examples) ---
    print("Creating label map from full dataset...")
    essential_labels = ['O', 'X', '[CLS]', '[SEP]']
    all_tags_in_data = set()
    for ex in all_examples:
         all_tags_in_data.update(ex.labels)

    unique_tags = sorted([tag for tag in all_tags_in_data if tag and tag not in essential_labels])
    all_unique_labels = essential_labels + unique_tags
    label_map = {label: i for i, label in enumerate(all_unique_labels)}
    idx2label = {i: label for label, i in label_map.items()}
    print(f"Label map created with {len(all_unique_labels)} labels from full data.")
    print(f"Labels: {all_unique_labels}")
    for lbl in essential_labels:
        if lbl not in label_map: print(f"FATAL: Essential label '{lbl}' missing!"); return False, None
    start_label_id = label_map['[CLS]']
    stop_label_id = label_map['[SEP]']

    # --- Separate Train/Val from All using Test GUIDs ---
    if not test_examples: print("Error: Received empty test_examples list."); return False, None
    test_guids = {ex.guid for ex in test_examples}
    train_val_examples = [ex for ex in all_examples if ex.guid not in test_guids]
    print(f"Separated data: {len(train_val_examples)} for Train/Val, {len(test_examples)} for Test.")
    del all_examples; gc.collect()

    if not train_val_examples: print("Error: No examples for training/validation."); return False, None

    # --- Split Train/Validation ---
    print(f"Splitting {len(train_val_examples)} examples into Train/Validation...")
    if config.val_size <= 0 or (1.0 - config.test_size <= 0):
        train_examples, dev_examples = train_val_examples, []
    else:
        relative_val_size = config.val_size / (1.0 - config.test_size) if (1.0 - config.test_size) > 0 else 0.1
        relative_val_size = max(0.01, min(0.99, relative_val_size))
        if len(train_val_examples) < 2:
            train_examples, dev_examples = train_val_examples, []
        else:
            try:
                stratify_labels = [ex.source for ex in train_val_examples] if len(set(ex.source for ex in train_val_examples)) > 1 else None
                train_examples, dev_examples = train_test_split(
                    train_val_examples, test_size=relative_val_size, random_state=config.seed, stratify=stratify_labels)
            except ValueError as split_err:
                 print(f"Warning: Could not stratify split ({split_err}). Splitting randomly.")
                 train_examples, dev_examples = train_test_split(
                    train_val_examples, test_size=relative_val_size, random_state=config.seed)

    print(f"Split complete: Train={len(train_examples)}, Val={len(dev_examples)}, Test={len(test_examples)}")
    del train_val_examples; gc.collect()
    if not train_examples: print("Error: No training examples after splitting."); return False, None

    # --- Load Tokenizer ---
    print(f"Loading tokenizer: {config.model_type}")
    try:
        tokenizer_kwargs = {'token': config.hf_token}
        if 'roberta' in config.model_type.lower(): tokenizer_kwargs['add_prefix_space'] = True
        tokenizer = AutoTokenizer.from_pretrained(config.model_type, **tokenizer_kwargs)
    except Exception as e: print(f"Error loading tokenizer {config.model_type}: {e}"); return False, None

    # --- Create Datasets & DataLoaders ---
    print("Creating Train/Dev Datasets...")
    train_dataset, dev_dataset = None, None
    try:
        train_dataset = NerDataset(train_examples, tokenizer, label_map, config.max_seq_length)
        del train_examples; gc.collect()
        if not train_dataset or len(train_dataset) == 0: print("Error: Training dataset is empty."); return False, None
        if dev_examples:
             dev_dataset = NerDataset(dev_examples, tokenizer, label_map, config.max_seq_length)
             del dev_examples; gc.collect()
             if not dev_dataset or len(dev_dataset) == 0: print("Warning: Validation dataset is empty."); dev_dataset = None
        else: dev_dataset = None
    except Exception as e: print(f"Error creating datasets: {e}"); return False, None

    print("Creating DataLoaders...")
    pin_memory = config.device == torch.device("cuda")
    train_dataloader, dev_dataloader = None, None
    try:
        train_dataloader = data.DataLoader(
            dataset=train_dataset, batch_size=config.batch_size, shuffle=True,
            num_workers=config.num_workers, collate_fn=NerDataset.pad, pin_memory=pin_memory, drop_last=True)
        if dev_dataset:
            dev_dataloader = data.DataLoader(
                dataset=dev_dataset, batch_size=config.batch_size, shuffle=False,
                num_workers=config.num_workers, collate_fn=NerDataset.pad, pin_memory=pin_memory)
    except Exception as e: print(f"Error creating DataLoaders: {e}"); return False, None

    # --- Training Setup ---
    if not train_dataloader or len(train_dataloader) == 0: print("Error: Training dataloader is empty."); return False, None
    effective_batch_size = config.batch_size * config.gradient_accumulation_steps
    num_update_steps_per_epoch = len(train_dataloader) // config.gradient_accumulation_steps
    total_train_steps = num_update_steps_per_epoch * config.total_train_epochs

    print("***** Training Information *****")
    print(f"  Model Type = {config.model_type}")
    print(f"  Num Training Examples = {len(train_dataset)}")
    print(f"  Num Validation Examples = {len(dev_dataset) if dev_dataset else 0}")
    print(f"  Num Test Examples = {len(test_examples)}") # Show test count
    print(f"  Num Epochs = {config.total_train_epochs}")
    print(f"  Total Optimization Steps = {total_train_steps}")
    print("******************************")

    # --- Initialize Model ---
    print("Initializing BERT_CRF_NER model...")
    model = None
    try:
        model = BERT_CRF_NER(
            model_type=config.model_type, start_label_id=start_label_id,
            stop_label_id=stop_label_id, num_labels=len(label_map),
            device=config.device, hf_token=config.hf_token)
        model.to(config.device)
        num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Model initialized with {num_params:,} trainable parameters.")
    except Exception as e: print(f"Error initializing model: {e}"); traceback.print_exc(); return False, None

    # --- Optimizer & Scheduler ---
    optimizer, scheduler = None, None
    try:
        param_optimizer = list(model.named_parameters())
        no_decay = ['bias', 'LayerNorm.bias', 'LayerNorm.weight']
        crf_fc_params_names = ['transitions', 'hidden2label.weight', 'hidden2label.bias']
        optimizer_grouped_parameters = [
            {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay) and not any(cf in n for cf in crf_fc_params_names) and p.requires_grad], 'weight_decay': config.weight_decay_finetune},
            {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay) and not any(cf in n for cf in crf_fc_params_names) and p.requires_grad], 'weight_decay': 0.0},
            {'params': [p for n, p in param_optimizer if n in ('transitions', 'hidden2label.weight') and p.requires_grad], 'lr': config.lr_crf_fc, 'weight_decay': config.weight_decay_crf_fc},
            {'params': [p for n, p in param_optimizer if n == 'hidden2label.bias' and p.requires_grad], 'lr': config.lr_crf_fc, 'weight_decay': 0.0}
        ]
        optimizer = optim.AdamW(optimizer_grouped_parameters, lr=config.learning_rate)
        if total_train_steps > 0:
            num_warmup_steps = int(total_train_steps * config.warmup_proportion)
            scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=total_train_steps)
            print(f"Scheduler created with {num_warmup_steps} warmup steps.")
        else: scheduler = None
    except Exception as e: print(f"Error setting up optimizer/scheduler: {e}"); traceback.print_exc(); return False, None

    # --- Training Loop ---
    global_step = 0
    best_valid_f1 = -1.0
    early_stopping_counter = 0
    best_epoch = 0
    # Updated history to store coverage
    history = {'train_loss': [], 'valid_f1': [], 'best_valid_f1': -1.0, 'epochs_ran': 0, 'valid_coverage': []}
    training_successful = True

    print("\n***** Starting Training *****")
    try:
        for epoch in range(config.total_train_epochs):
            model.train()
            tr_loss = 0.0; nb_tr_steps = 0
            epoch_start_time = time.time()
            train_iterator = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{config.total_train_epochs}", leave=False)

            for step, batch in enumerate(train_iterator):
                try:
                    # Unpack batch (ignore index tensor during training)
                    input_ids, input_mask, segment_ids, predict_mask, label_ids, _ = batch
                    input_ids, input_mask, segment_ids, predict_mask, label_ids = \
                        input_ids.to(config.device), input_mask.to(config.device), segment_ids.to(config.device), \
                        predict_mask.to(config.device), label_ids.to(config.device)

                    loss = model.neg_log_likelihood(input_ids, segment_ids, input_mask, label_ids)
                    # Check for NaN/Inf loss *before* potential division
                    if torch.isnan(loss) or torch.isinf(loss):
                        print(f"\nWarning: NaN/Inf loss detected at Epoch {epoch+1}, Step {step}. Skipping batch update.")
                        optimizer.zero_grad() # Still zero grad
                        continue

                    if config.gradient_accumulation_steps > 1: loss = loss / config.gradient_accumulation_steps

                    loss.backward()
                    tr_loss += loss.item() * config.gradient_accumulation_steps

                    if (step + 1) % config.gradient_accumulation_steps == 0:
                         torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
                         optimizer.step()
                         if scheduler: scheduler.step()
                         optimizer.zero_grad()
                         global_step += 1; nb_tr_steps += 1
                         if nb_tr_steps > 0: train_iterator.set_postfix({'loss': f'{tr_loss / nb_tr_steps:.4f}'})

                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        print(f"\nWARNING: CUDA OOM at Epoch {epoch+1}, Step {step}. Skipping batch.")
                        optimizer.zero_grad(); torch.cuda.empty_cache(); gc.collect()
                        continue
                    else: print(f"\nRuntime error during training step: {e}"); traceback.print_exc(); raise e

            # --- End of Epoch ---
            epoch_end_time = time.time()
            epoch_duration_mins = (epoch_end_time - epoch_start_time) / 60.0
            history['epochs_ran'] = epoch + 1
            avg_train_loss = tr_loss / nb_tr_steps if nb_tr_steps > 0 else float('nan')
            history['train_loss'].append(avg_train_loss)
            print(f"\nEpoch {epoch+1} Summary: Avg Train Loss={avg_train_loss:.4f}, Time={epoch_duration_mins:.2f}m")

            # --- Validation ---
            current_valid_f1 = -1.0
            current_coverage = {}
            if dev_dataloader:
                print("Running Validation...")
                try:
                    # Evaluate returns coverage metrics as the third item now
                    current_valid_f1, _, current_coverage = evaluate(model, dev_dataloader, epoch + 1, 'Validation Set', label_map, idx2label, config.device)
                    history['valid_f1'].append(current_valid_f1)
                    history['valid_coverage'].append(current_coverage) # Store coverage
                    print(f"Validation F1: {current_valid_f1:.4f}")
                    print(f"  Avg Pred Entities/Sent: {current_coverage.get('avg_pred_entities_per_sentence', 0.0):.3f}")

                    if current_valid_f1 > best_valid_f1:
                         print(f"  New best validation F1! ({current_valid_f1:.4f} > {best_valid_f1:.4f}). Saving model...")
                         best_valid_f1 = current_valid_f1; best_epoch = epoch + 1
                         history['best_valid_f1'] = best_valid_f1
                         early_stopping_counter = 0

                         checkpoint = {
                             'epoch': best_epoch, 'model_state': model.state_dict(),
                             'optimizer_state': optimizer.state_dict(), 'valid_f1': best_valid_f1,
                             'config': vars(config), 'label_map': label_map, 'idx2label': idx2label,
                             'max_seq_length': config.max_seq_length, 'model_type': config.model_type
                         }
                         if scheduler: checkpoint['scheduler_state'] = scheduler.state_dict()
                         best_model_save_path = os.path.join(model_base_dir, 'best_model.pt')
                         torch.save(checkpoint, best_model_save_path)
                         tokenizer.save_pretrained(model_base_dir)
                         print(f"  Best model and tokenizer saved to {model_base_dir}")
                    else:
                         early_stopping_counter += 1
                         print(f"  Validation F1 ({current_valid_f1:.4f}) did not improve. Counter: {early_stopping_counter}/{config.early_stopping_patience}")

                except Exception as eval_err:
                     print(f"\nError during validation for Epoch {epoch+1}: {eval_err}")
                     traceback.print_exc()
                     history['valid_f1'].append(-1.0); history['valid_coverage'].append({})

                if config.early_stopping_patience > 0 and early_stopping_counter >= config.early_stopping_patience:
                     print(f"\nEarly stopping triggered after {epoch+1} epochs.")
                     break

            else: # No validation
                best_epoch = epoch + 1
                if (epoch + 1) % config.checkpoint_freq == 0 or (epoch + 1) == config.total_train_epochs:
                    print(f"Saving model checkpoint at epoch {epoch+1} (no validation set).")
                    checkpoint = { 'epoch': best_epoch, 'model_state': model.state_dict(), # ... (rest of checkpoint data) ...
                                   'optimizer_state': optimizer.state_dict(), 'valid_f1': -1.0, 'config': vars(config),
                                   'label_map': label_map, 'idx2label': idx2label, 'max_seq_length': config.max_seq_length,
                                   'model_type': config.model_type }
                    if scheduler: checkpoint['scheduler_state'] = scheduler.state_dict()
                    chk_path = os.path.join(model_base_dir, 'best_model.pt')
                    torch.save(checkpoint, chk_path)
                    tokenizer.save_pretrained(model_base_dir)
                    print(f"  Checkpoint and tokenizer saved to {model_base_dir}")

            # --- Save History ---
            history_path = os.path.join(model_base_dir, 'training_history.json')
            try:
                def default_serializer_hist(obj): # Need robust serializer
                    if isinstance(obj, (np.floating, float)): return float(obj)
                    if isinstance(obj, (np.integer, int)): return int(obj)
                    if isinstance(obj, np.ndarray): return obj.tolist()
                    if isinstance(obj, dict): return {k: default_serializer_hist(v) for k, v in obj.items()}
                    return str(obj)
                with open(history_path, 'w') as f: json.dump(history, f, indent=2, default=default_serializer_hist)
            except Exception as e: print(f"Warning: Could not save training history: {e}")

            gc.collect(); torch.cuda.empty_cache()

    except Exception as train_err:
        print(f"\n\n>>> FATAL ERROR during training loop at Epoch {history['epochs_ran']}! Error: {train_err} <<<")
        traceback.print_exc(); training_successful = False

    # === End of Training ===
    print("\n***** Training Finished *****")
    if dev_dataloader: print(f"Best validation F1: {best_valid_f1:.4f} at epoch {best_epoch}")
    else: print(f"Training completed after {history['epochs_ran']} epochs (no validation).")

    if not training_successful:
        print("Skipping final evaluation due to training failure.")
        # Clean up resources
        if 'model' in locals() and model: del model
        if 'train_dataloader' in locals(): del train_dataloader
        if 'dev_dataloader' in locals(): del dev_dataloader
        if 'train_dataset' in locals(): del train_dataset
        if 'dev_dataset' in locals(): del dev_dataset
        gc.collect(); torch.cuda.empty_cache()
        return False, None

    # === Final Evaluation ===
    print("\n***** Final Evaluation on Test Set *****")
    best_model_path = os.path.join(model_base_dir, 'best_model.pt')
    if not os.path.exists(best_model_path):
        print(f"Error: Best model checkpoint ('{best_model_path}') not found."); return False, None

    print(f"Loading best model for final evaluation from: {best_model_path}")
    final_model, final_tokenizer = None, None
    final_results = {}
    try:
        checkpoint = torch.load(best_model_path, map_location=config.device, weights_only=False) # Use weights_only=False
        loaded_config_dict = checkpoint.get('config', vars(config))
        loaded_model_type = checkpoint.get('model_type', config.model_type)
        label_map = checkpoint['label_map']; idx2label = checkpoint['idx2label']
        num_labels_loaded = len(label_map); start_label_id = label_map['[CLS]']; stop_label_id = label_map['[SEP]']

        final_model = BERT_CRF_NER(model_type=loaded_model_type, start_label_id=start_label_id, stop_label_id=stop_label_id,
                                   num_labels=num_labels_loaded, device=config.device, hf_token=config.hf_token)
        final_model.load_state_dict(checkpoint['model_state'])
        final_model.to(config.device); final_model.eval()
        final_tokenizer = AutoTokenizer.from_pretrained(model_base_dir, token=config.hf_token)
        print("Best model and tokenizer loaded.")

    except Exception as e: print(f"Error loading best model/tokenizer: {e}"); traceback.print_exc(); return False, None

    # --- Prepare Test Dataloader ---
    print("Preparing Test Dataset/Loader...")
    if not test_examples: print("Error: test_examples list is empty."); return False, None
    test_dataset, test_dataloader = None, None
    try:
        test_dataset = NerDataset(test_examples, final_tokenizer, label_map, config.max_seq_length)
        if not test_dataset or len(test_dataset) == 0: print("Error: Test dataset empty."); return False, None
        test_dataloader = data.DataLoader(dataset=test_dataset, batch_size=config.batch_size, shuffle=False,
                                         num_workers=config.num_workers, collate_fn=NerDataset.pad, pin_memory=pin_memory)
        if not test_dataloader or len(test_dataloader) == 0: print("Error: Test dataloader empty."); del test_dataset; gc.collect(); return False, None
    except Exception as e: print(f"Error creating test dataset/loader: {e}"); return False, None

    # --- 1. Overall Test Set Evaluation ---
    print("\n--- Evaluating on Overall Test Set ---")
    overall_test_f1, overall_class_metrics, overall_coverage_results = 0.0, {}, {}
    try:
        overall_test_f1, overall_class_metrics, overall_coverage_results = evaluate(
            final_model, test_dataloader, "Final Overall", 'Test Set (Overall)', label_map, idx2label, config.device)
        print(f"Overall Test F1 Score: {overall_test_f1:.4f}")
        print(f"Overall Avg Pred Entities/Sent: {overall_coverage_results.get('avg_pred_entities_per_sentence', 0.0):.3f}")
    except Exception as e: print(f"Error during overall test evaluation: {e}"); traceback.print_exc()

    # Store final results including coverage
    final_results = {
        'model_type': loaded_model_type, 'best_epoch': best_epoch, 'best_valid_f1': best_valid_f1,
        'overall_test_f1': overall_test_f1, 'overall_class_metrics': {},
        'overall_coverage_metrics': overall_coverage_results, # Store coverage dict
        'source_specific_test_results': {}
    }
    for cls_id, metrics in overall_class_metrics.items():
        label = metrics.get('label', f"Unknown_{cls_id}")
        final_results['overall_class_metrics'][label] = {k: v for k, v in metrics.items() if k != 'label'}

    # --- 2. Source-Specific Test Evaluation ---
    print("\n--- Evaluating on Source-Specific Test Subsets ---")
    # Prepare source-specific loaders ONCE
    source_datasets = {}
    source_dataloaders = {}
    print("Preparing source-specific test datasets/loaders...")
    for source_name in SOURCES_TO_EVALUATE:
        source_examples = [ex for ex in test_examples if ex.source == source_name]
        if not source_examples: print(f"  No examples for source '{source_name}'."); continue
        try:
            s_dataset = NerDataset(source_examples, final_tokenizer, label_map, config.max_seq_length)
            if not s_dataset or len(s_dataset)==0: print(f"  Warn: Dataset empty for {source_name}"); continue
            s_dataloader = data.DataLoader(s_dataset, batch_size=config.batch_size, shuffle=False,
                                           num_workers=config.num_workers, collate_fn=NerDataset.pad, pin_memory=pin_memory)
            if not s_dataloader or len(s_dataloader)==0: print(f"  Warn: Dataloader empty for {source_name}"); del s_dataset; continue
            source_datasets[source_name] = s_dataset; source_dataloaders[source_name] = s_dataloader
            print(f"  Prepared loader for {source_name} ({len(s_dataset)} examples)")
        except Exception as e: print(f"  Error preparing data for source {source_name}: {e}"); traceback.print_exc()

    # Evaluate using prepared loaders
    for source_name in SOURCES_TO_EVALUATE:
        if source_name not in source_dataloaders:
             final_results['source_specific_test_results'][source_name] = {'f1': 0.0, 'class_metrics': {}, 'coverage_metrics': {}, 'message': f'No data/loader for {source_name}'}
             continue

        print(f"\n--- Evaluating source: {source_name} ---")
        s_dataloader = source_dataloaders[source_name]
        source_f1, source_class_metrics, source_coverage_results = 0.0, {}, {}
        try:
            source_f1, source_class_metrics, source_coverage_results = evaluate(
                final_model, s_dataloader, f"Final {source_name}", f'Test Set ({source_name})', label_map, idx2label, config.device)
            print(f"  F1 Score for {source_name}: {source_f1:.4f}")
            print(f"  Avg Pred Entities/Sent ({source_name}): {source_coverage_results.get('avg_pred_entities_per_sentence', 0.0):.3f}")

            final_results['source_specific_test_results'][source_name] = {
                'f1': source_f1, 'class_metrics': {}, 'coverage_metrics': source_coverage_results }
            for cls_id, metrics in source_class_metrics.items():
                label = metrics.get('label', f"Unknown_{cls_id}")
                final_results['source_specific_test_results'][source_name]['class_metrics'][label] = {k: v for k, v in metrics.items() if k != 'label'}

        except Exception as e:
            print(f"  Error during evaluation for source {source_name}: {e}"); traceback.print_exc()
            final_results['source_specific_test_results'][source_name] = {'f1': 0.0, 'class_metrics': {}, 'coverage_metrics': {}, 'message': f'Evaluation error: {e}'}
        finally:
             del source_dataloaders[source_name]; del source_datasets[source_name]
             gc.collect(); torch.cuda.empty_cache()

    # --- Save Final Combined Results ---
    results_path = os.path.join(model_base_dir, 'final_evaluation_results.json')
    print(f"\nSaving final combined evaluation results to {results_path}")
    try:
        # Use robust serializer
        def default_serializer_final(obj):
            if isinstance(obj, np.integer): return int(obj)
            elif isinstance(obj, np.floating): return float(obj)
            elif isinstance(obj, np.ndarray): return obj.tolist()
            elif isinstance(obj, (np.bool_, bool)): return bool(obj)
            if isinstance(obj, dict): return {k: default_serializer_final(v) for k, v in obj.items()}
            if isinstance(obj, list): return [default_serializer_final(i) for i in obj]
            try: return json.JSONEncoder.default(None, obj)
            except TypeError: return str(obj)
        with open(results_path, 'w', encoding='utf-8') as f:
            json.dump(final_results, f, indent=2, default=default_serializer_final, ensure_ascii=False)
        print("Final results saved successfully.")
    except Exception as e: print(f"Error saving final results to JSON: {e}"); traceback.print_exc()

    # --- Example Prediction ---
    print("\n--- Example Prediction ---")
    sample_text = "The threat actor APT29 deployed Cobalt Strike beacons after compromising credentials via a phishing email."
    print(f"Input text: \"{sample_text}\"")
    try:
        predicted_entities = predict_entities(final_model, final_tokenizer, sample_text, label_map, idx2label, config.max_seq_length, config.device)
        print("Predicted Entities:")
        if predicted_entities:
            for entity in predicted_entities: print(f"  - {entity}")
        else: print("  No entities predicted.")
    except Exception as pred_err: print(f"Error during example prediction: {pred_err}")
    print("-------------------------")

    print(f"===== Finished Training Run for Model: {config.model_type} =====")

    # --- Final Cleanup ---
    del final_model, final_tokenizer, model, optimizer, checkpoint
    if scheduler: del scheduler
    del train_dataloader, test_dataloader
    if dev_dataloader: del dev_dataloader
    del train_dataset, test_dataset
    if dev_dataset: del dev_dataset
    gc.collect(); torch.cuda.empty_cache()

    return True, final_results
# -

# ## 9. Training Loop (Iterating Models - Original Data + Coverage)

# +
print("Starting training loop for specified models (Original Data + Coverage)...")

all_models_results = {}
training_summary = {}

# --- Get base configuration ---
base_config_dict = {
    'dataset_path': DATASET_PATH, 'test_size': TEST_SIZE, 'val_size': VAL_SIZE,
    'max_seq_length': MAX_SEQ_LENGTH, 'batch_size': BATCH_SIZE, 'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
    'epochs': EPOCHS, 'learning_rate': LEARNING_RATE, 'lr_crf_fc': LR_CRF_FC,
    'weight_decay_finetune': WEIGHT_DECAY_FINETUNE, 'weight_decay_crf_fc': WEIGHT_DECAY_CRF_FC,
    'warmup_proportion': WARMUP_PROPORTION, 'max_grad_norm': MAX_GRAD_NORM,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE, 'checkpoint_freq': CHECKPOINT_FREQ,
    'num_workers': NUM_WORKERS, 'output_dir': BASE_OUTPUT_DIR, 'seed': SEED,
    'device': DEVICE, 'hf_token': HF_TOKEN
}

# --- Pre-load and split data ONCE (Original approach) ---
print("\n--- Pre-loading and Splitting Original Data ONCE ---")
test_examples_global = None
try:
    df_main = pd.read_csv(base_config_dict['dataset_path'], keep_default_na=False, encoding='utf-8', on_bad_lines='warn', dtype=str)
    # Use the original prepare function (no deduplication)
    all_examples_main = prepare_unified_dataset(df_main)
    del df_main; gc.collect()
    if not all_examples_main: raise ValueError("No examples created from main dataset.")

    if not (0 < base_config_dict['test_size'] < 1): raise ValueError("Invalid TEST_SIZE.")
    if len(all_examples_main) < 2: raise ValueError("Insufficient examples for split.")

    # Split the original (potentially duplicated) examples
    try:
        stratify_labels = [ex.source for ex in all_examples_main] if len(set(ex.source for ex in all_examples_main)) > 1 else None
        _ , test_examples_global = train_test_split(
            all_examples_main, test_size=base_config_dict['test_size'],
            random_state=base_config_dict['seed'], stratify=stratify_labels)
    except ValueError as split_err:
         print(f"Warning: Could not stratify split ({split_err}). Splitting randomly.")
         _ , test_examples_global = train_test_split(
            all_examples_main, test_size=base_config_dict['test_size'], random_state=base_config_dict['seed'])

    print(f"Global test set created: {len(test_examples_global)} examples")
    del all_examples_main; gc.collect()

except FileNotFoundError:
    print(f"FATAL: Dataset file not found: {base_config_dict['dataset_path']}")
    MODELS_TO_TRAIN = []
except Exception as data_err:
    print(f"FATAL: Failed to prepare global data split: {data_err}")
    traceback.print_exc(); MODELS_TO_TRAIN = []

# --- Iterate through models ---
for model_id in MODELS_TO_TRAIN:
    print(f"\n{'='*25} Processing Model: {model_id} {'='*25}")
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()

    current_config_dict = base_config_dict.copy()
    current_config_dict['model_type'] = model_id
    # Output dir for this model run is set within the train function now
    config_run = Config(current_config_dict)

    success = False; results = None
    if test_examples_global is not None:
        try:
             # Pass the original (potentially duplicated) test set
             success, results = train(config_run, test_examples_global)
             if success and results: all_models_results[model_id] = results
             elif success: print(f"Warn: {model_id} success but no results."); success = False
        except Exception as e:
             print(f"!!! UNCAUGHT ERROR during train() for {model_id}: {e} !!!"); traceback.print_exc(); success = False
    else:
        print(f"Skipping training for {model_id} due to data prep failure."); success = False

    training_summary[model_id] = "Success" if success else "Failed"
    print(f"Finished processing {model_id}. Status: {training_summary[model_id]}")
    print(f"{'='*60}"); time.sleep(2)

# --- Final Summary ---
print("\n===== Overall Training Loop Summary (Original Data + Coverage) =====")
if not training_summary: print("No models were processed.")
else:
    for model_id, status in training_summary.items():
        print(f"Model: {model_id:<30} | Status: {status}")
        if status == "Success" and model_id in all_models_results:
             res = all_models_results[model_id]
             print(f"  -> Best Valid F1: {res.get('best_valid_f1', 'N/A'):.4f} (Epoch {res.get('best_epoch', 'N/A')})")
             print(f"  -> Overall Test F1: {res.get('overall_test_f1', 'N/A'):.4f}")
             # Print key coverage metric
             coverage = res.get('overall_coverage_metrics', {})
             print(f"  -> Overall Avg Pred Entities/Sent: {coverage.get('avg_pred_entities_per_sentence', 'N/A'):.3f}")
             print(f"  -> Overall Sentence Entity Recall (Micro): {100*coverage.get('sentence_entity_recall_micro', 0.0):.2f}%")

print("====================================================================")

# Save aggregated results
summary_results_path = os.path.join(BASE_OUTPUT_DIR, 'all_models_summary_results_coverage.json')
print(f"\nSaving aggregated results to {summary_results_path}")
try:
    # Use robust serializer again
    def default_serializer_agg(obj):
            if isinstance(obj, np.integer): return int(obj)
            elif isinstance(obj, np.floating): return float(obj)
            elif isinstance(obj, np.ndarray): return obj.tolist()
            elif isinstance(obj, (np.bool_, bool)): return bool(obj)
            if isinstance(obj, dict): return {k: default_serializer_agg(v) for k, v in obj.items()}
            if isinstance(obj, list): return [default_serializer_agg(i) for i in obj]
            try: return json.JSONEncoder.default(None, obj)
            except TypeError: return str(obj)
    with open(summary_results_path, 'w', encoding='utf-8') as f:
        json.dump(all_models_results, f, indent=2, default=default_serializer_agg, ensure_ascii=False)
    print("Aggregated results saved successfully.")
except Exception as e: print(f"Error saving aggregated results: {e}")

print("\nScript finished.")
# -

# ## 10. Conclusion
#
# The notebook has finished iterating through the specified models using the **original dataset (potentially containing duplicates)**. The evaluation now includes standard P/R/F1 metrics as well as **sentence-level coverage metrics** (like average predicted entities per sentence and sentence-level recall) to better assess the model's ability to capture information breadth. Check the output directories within `BASE_OUTPUT_DIR` for saved models, tokenizers, training history, and detailed evaluation results (`final_evaluation_results.json`) for each successful run. An aggregated summary (`all_models_summary_results_coverage.json`) is also saved in the base output directory.

2025-05-01 18:46:28.851594: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746121588.866744  291087 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746121588.871393  291087 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-01 18:46:28.887697: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Using device: cuda
Starting training loop for specified models (Original Data + Coverage)...

--- Pre-loading and Splitting Original Data ONCE ---
--- Starting Data Preparation (No Deduplication) ---
Creating globally unique sentence IDs (Source_SentenceID)...
Unique instance IDs created.
Found 23477 initial sentence instances based on Unique_Sentence_ID.
Creating InputExamples...


Preparing Examples:   0%|          | 0/23477 [00:00<?, ?it/s]

Created 23477 InputExamples.
Data preparation took 2.57 seconds.
--- Data Preparation Finished ---
Global test set created: 3522 examples

========================= Processing Model: bert-base-cased =========================

===== Starting Training Run for Model: bert-base-cased =====
Output Directory: /home/yasir.ech-chammakhy/lustre/cyber_cc-lcbfvhtc9qm/users/yasir.ech-chammakhy/BEACON/outputs_beacon_coverage/bert-base-cased_unified_coverage
Loading full unified dataset from /home/yasir.ech-chammakhy/lustre/cyber_cc-lcbfvhtc9qm/users/yasir.ech-chammakhy/BEACON/dataset/beacon_stix_v1.csv (for train/dev split)...
--- Starting Data Preparation (No Deduplication) ---
Creating globally unique sentence IDs (Source_SentenceID)...
Unique instance IDs created.
Found 23477 initial sentence instances based on Unique_Sentence_ID.
Creating InputExamples...


Preparing Examples:   0%|          | 0/23477 [00:00<?, ?it/s]

...


--- Evaluation Results (Test Set (CyNER) at Final CyNER) ---
Eval Time: 0.040 minutes, Sentences Evaluated: 656
--- Overall Performance (Token Level) ---
Overall Precision: 77.39%
Overall Recall: 79.71%
Overall F1-Score: 78.53%
--- Coverage Metrics (Sentence Level) ---
Avg Predicted Entities / Sentence: 1.267
Avg Predicted Entity Types / Sentence: 0.808
Sentence Entity Recall (Macro Avg): 82.41%
Sentence Type Recall (Macro Avg): 84.45%
Sentence Entity Recall (Micro Avg - Total Correct/Total True): 76.39%
--------------------------------------------------------------
Per-class metrics (P, R, F1, Support - based on overall tokens):
  B-Attack-Pattern    : P=0.0000, R=0.0000, F1=0.0000, S=0
  B-Domain-Name       : P=0.0000, R=0.0000, F1=0.0000, S=0
  B-File              : P=0.0000, R=0.0000, F1=0.0000, S=0
  B-Identity          : P=0.5800, R=0.8286, F1=0.6824, S=70
  B-Indicator         : P=0.9426, R=0.7787, F1=0.8528, S=253
  B-Location          : P=0.0000, R=0.0000, F1=0.0000, S=0
  B-

Evaluating on Test Set (Attacker):   0%|          | 0/24 [00:00<?, ?it/s]


--- Evaluation Results (Test Set (Attacker) at Final Attacker) ---
Eval Time: 0.023 minutes, Sentences Evaluated: 372
--- Overall Performance (Token Level) ---
Overall Precision: 79.48%
Overall Recall: 56.69%
Overall F1-Score: 66.18%
--- Coverage Metrics (Sentence Level) ---
Avg Predicted Entities / Sentence: 2.438
Avg Predicted Entity Types / Sentence: 1.653
Sentence Entity Recall (Macro Avg): 51.80%
Sentence Type Recall (Macro Avg): 55.01%
Sentence Entity Recall (Micro Avg - Total Correct/Total True): 55.65%
--------------------------------------------------------------
Per-class metrics (P, R, F1, Support - based on overall tokens):
  B-Attack-Pattern    : P=0.5882, R=0.4688, F1=0.5217, S=128
  B-Campaign          : P=0.5556, R=0.3030, F1=0.3922, S=33
  B-Course-of-Action  : P=0.7812, R=0.5000, F1=0.6098, S=50
  B-File              : P=0.0000, R=0.0000, F1=0.0000, S=0
  B-Identity          : P=0.7128, R=0.6837, F1=0.6979, S=196
  B-Indicator         : P=0.8000, R=0.3478, F1=0.4848,

Evaluating on Test Set (DNRTI):   0%|          | 0/62 [00:00<?, ?it/s]


--- Evaluation Results (Test Set (DNRTI) at Final DNRTI) ---
Eval Time: 0.051 minutes, Sentences Evaluated: 987
--- Overall Performance (Token Level) ---
Overall Precision: 79.93%
Overall Recall: 80.16%
Overall F1-Score: 80.05%
--- Coverage Metrics (Sentence Level) ---
Avg Predicted Entities / Sentence: 3.404
Avg Predicted Entity Types / Sentence: 2.268
Sentence Entity Recall (Macro Avg): 73.53%
Sentence Type Recall (Macro Avg): 77.72%
Sentence Entity Recall (Micro Avg - Total Correct/Total True): 77.32%
--------------------------------------------------------------
Per-class metrics (P, R, F1, Support - based on overall tokens):
  B-Attack-Pattern    : P=0.8373, R=0.7512, F1=0.7919, S=418
  B-Domain-Name       : P=0.0000, R=0.0000, F1=0.0000, S=0
  B-File              : P=0.0000, R=0.0000, F1=0.0000, S=0
  B-IPv4-Addr         : P=0.0000, R=0.0000, F1=0.0000, S=0
  B-Identity          : P=0.8993, R=0.9058, F1=0.9025, S=690
  B-Location          : P=0.9093, R=0.9525, F1=0.9304, S=400
 